In [1]:
import pyautogui
import time
import openpyxl
from datetime import date , datetime , timedelta
from pathlib import Path
import pandas as pd
import locale
import time
import sys
import os
import glob
import pyperclip
from openpyxl import load_workbook
import shutil
import schedule
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options

In [2]:
# Definir o locale para português do Brasil
locale.setlocale(locale.LC_TIME, 'Portuguese_Brazil.1252')

# Definir o caminho de destino
destination_folder = os.path.join(
    os.getcwd(),
     r"\\fscds01\Groups_CDA\Tra\@Ger. Adm. Transportes\28. Rel_Automacoes_BackOffice\Multi_Embarcador\Cargas",
    (datetime.now() - timedelta(days=1)).strftime("%Y"),
    datetime.now().strftime("%B")
)


# Converter barras invertidas para barras normais para evitar problemas
destination_folder = os.path.normpath(destination_folder)

# Criar os diretórios necessários se não existirem
os.makedirs(destination_folder, exist_ok=True)


# Configurar as opções do Chrome para definir o local de download
chrome_options = webdriver.ChromeOptions()
prefs = {
    "download.default_directory": destination_folder,
    "download.directory_upgrade": True,
    "download.prompt_for_download": False,
    "safebrowsing.enabled": True
}
chrome_options.add_experimental_option("prefs", prefs)

# Inicializar o WebDriver
navegador = webdriver.Chrome(options=chrome_options)

In [3]:
# Listar todos os arquivos dentro do diretório de destino
file_list = glob.glob(os.path.join(destination_folder, "*"))

In [4]:
# Excluir cada arquivo encontrado
for file_path in file_list:
    os.remove(file_path)

In [5]:
def preencher_primeiro_dia_mes():
    
    #Criar data incio do mês
    data_anterior = datetime.now() - timedelta(days=1)
    Inicio_mes = pd.Period(data_anterior, freq = "M").start_time.date()
    pyautogui.write(Inicio_mes.strftime("%d/%m/%y"))

In [6]:
def preencher_data_final():
   
    # Criar data anterior, visão D-1
    data_anterior = datetime.now() - timedelta(days=1)
    pyautogui.write(data_anterior.strftime("%d/%m/%y"))

In [7]:
def click_on_image(image_name, confidence=0.8, double_click=False):
    while True:
        image_location = pyautogui.locateCenterOnScreen(image_name, confidence=confidence)
        if image_location:
            if double_click:
                pyautogui.doubleClick(image_location.x, image_location.y)
            else:
                pyautogui.click(image_location.x, image_location.y)
            break

In [8]:
pyautogui.PAUSE = 0.5

#Passo 1: Entrar no Chrome
#navegador = webdriver.Chrome()
navegador.get("https://grupocarrefour.multiembarcador.com.br/Login")

time.sleep(3)

# Passo 2: Fazer o login no Multi Embarcador
navegador.find_element(By.XPATH, '/html/body/div[2]/div/div/div[2]/div/div[1]/div[2]/div/form/div[2]/div[1]/input').send_keys('maisa_santos')
navegador.find_element(By.XPATH, '/html/body/div[2]/div/div/div[2]/div/div[1]/div[2]/div/form/div[2]/div[2]/input').send_keys('Carrefour@26')
navegador.find_element(By.XPATH, '/html/body/div[2]/div/div/div[2]/div/div[1]/div[2]/div/form/div[2]/div[3]/button').click()

time.sleep(8)

In [9]:
#Ir no caminho de Cargas
click_on_image('Caminho_link.PNG')
pyautogui.write("https://grupocarrefour.multiembarcador.com.br/#Relatorios/Cargas/Carga")
pyautogui.press("enter")

time.sleep(8)

In [10]:
#Clicar na Data Inicio
click_on_image('DataInicial.PNG', confidence=0.8)

In [11]:
# Chamando a função
preencher_primeiro_dia_mes()

In [12]:
#Clicar na Data final

pyautogui.hotkey('tab')

In [13]:
#Chamando a função
preencher_data_final()

In [14]:
time.sleep(3)

#Clicar na tela só para fechar o calendario

navegador.find_element(By.XPATH, '/html/body/div[2]/div/div/main/section/div[2]/div/div[1]').click()

#Pesquisar o tipo de relatório
navegador.find_element(By.XPATH, '/html/body/div[2]/div/div/main/section/div[2]/div/div[2]/form/div[1]/div[1]/div[6]/div/div/div/button').click()
time.sleep(1)
pyautogui.write('Giro')
pyautogui.press("enter")
#Selecionar o tipo de relatório
navegador.find_element(By.XPATH, '/html/body/div[2]/div/div/main/div/div/div/div[2]/div[3]/div[1]/div/div/table/tbody/tr[1]/td[3]/a').click()
time.sleep(1)

#Aguardar carregar a página

while len(navegador.find_elements(By.XPATH, '//*[@id="knockoutCRUDConfiguracaoRelatorio"]')) <1:
    time.sleep(1)
time.sleep(1) #garantia

#Apertar em Preview
navegador.find_element(By.XPATH, '/html/body/div[2]/div/div/main/section/div[2]/div/div[2]/div[1]/div[2]/button[2]').click()

while len(navegador.find_elements(By.XPATH, '//*[@id="container-colunas-gridPreviewRelatorio"]/div[1]/label')) <1:
    time.sleep(1)
time.sleep(1) #garantia

#Rodar a Página
pyautogui.hotkey("pagedown")

#Rodar a Página
pyautogui.hotkey("pagedown")

#Rodar a Página
pyautogui.hotkey("pagedown")

#Rodar a Página
pyautogui.hotkey("pagedown")

#Rodar a Página
pyautogui.hotkey("pagedown")

#Rodar a Página
pyautogui.hotkey("pagedown")

#Gerar relatório
navegador.find_element(By.XPATH, '/html/body/div[2]/div/div/main/section/div[2]/div/div[2]/div[4]/button[1]/span').click()



In [15]:
# Aguardar o término do download
while True:
    time.sleep(5)  # Aguarde 5 segundos antes de verificar novamente

    # Verificar se há arquivos no diretório de destino
    if not any(file.endswith(".csv") for file in os.listdir(destination_folder)):
        # Se não houver arquivos, o download ainda está em progresso
        continue
    else:
        # Se o diretório não estiver vazio, o download foi concluído
        # Verificar se há um arquivo com a extensão .csv no diretório
        csv_files = glob.glob(os.path.join(destination_folder, "*.csv"))
        
        if csv_files:
            # O arquivo CSV foi baixado
            downloaded_file = csv_files[0]  # Assume que o primeiro arquivo encontrado é o desejado
            break

# Criar a estrutura de diretórios para a segunda pasta com ano e mês
second_destination_folder = os.path.join(
    r"\\fscds01\Groups_RS\Arq_rms\25.Rel_Automacoes_CDRS\MultiEmbarcador\Rel.Carga",
    (datetime.now() - timedelta(days=1)).strftime("%Y"),
    datetime.now().strftime("%B")
)
os.makedirs(second_destination_folder, exist_ok=True)

# Caminho completo do arquivo na segunda pasta
destination_file_path = os.path.join(second_destination_folder, os.path.basename(downloaded_file))

# Listar todos os arquivos dentro do diretório de destino
file_list = glob.glob(os.path.join(second_destination_folder, "*"))

# Excluir cada arquivo encontrado
for file_path in file_list:
    os.remove(file_path)

# Copiar o arquivo para a segunda pasta (substituindo se já existir)
shutil.copy(downloaded_file, destination_file_path)

time.sleep(5)

# Fechar o navegador após o término do download
navegador.quit()